# Coronspec Tools example notebook

This notebook demonstrates how to use the coronspec tools library to work with HST-17092 data. It's a work-in-progress that will evolve as the coronspec library is built out.

At the end of this notebook, we will print the inferred positions of the occulted primaries.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
import matplotlib as mpl
from matplotlib import pyplot as plt

In [4]:
mpl.rcParams.update({'image.aspect':  'auto', 'image.origin': 'lower'})

In [5]:
from coronspec_tools import utils as ctutils
from coronspec_tools import misc as ctmisc

## List and organize the data files

I like to use a Pandas dataframe to organize my data files by metadata. Since all data files of the same type (e.g. sx1, sx2, flt, et cetera) have the same header keywords, they fit neatly into a dataframe format where the columns represent the keyword value and each row is a separate file. `coronspec_tools.utils` has some functions for setting this up.

In [6]:
# First, let's list all the data files available. Set your path as appropriate.
data_files = sorted(Path("../../../data/MAST_2025-12-16T2001/HST/").glob("of*/*fits"))

In [7]:
# here are all the files:
for i in data_files:
    print(i)

../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_asn.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_crj.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_drj.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_flt.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_raw.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_sx1.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_sx2.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_wav.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_asn.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_crj.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_drj.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_flt.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_raw.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020/of0i01020_sx2.fits
../../../data/MAST_2025-12-16T2001/HST/of0i01020

In [8]:
# for each file type, let's make a separate "file manager" dataframe using utils.organize_files_by_header(list_of_files)
file_managers = {}
for f in data_files:
    ftype = f.stem.split("_")[1]
    if ftype not in file_managers.keys():
        file_managers[ftype] = []
    file_managers[ftype].append(f)
for ft in file_managers:
    file_managers[ft] = ctutils.organize_files_by_header(file_managers[ft])

In [9]:
print("The available filetypes are:", ', '.join(sorted(file_managers.keys())))

The available filetypes are: asn, blv, crj, drj, flt, raw, sx1, sx2, wav


In [10]:
asn_files = file_managers.pop("asn")

## Defringing
See https://stistools.readthedocs.io/en/latest/defringe_guide.html

In [11]:
from coronspec_tools import defringing_tools
import stistools

The following tasks in the stistools package can be run with TEAL:
   basic2d      calstis     ocrreject     wavecal        x1d          x2d


/Users/jaguilar/Projects/miniconda3/envs/stiscoron/lib/python3.13/site-packages/stsci/tools/nmpfit.py:8: UserWarning: NMPFIT is deprecated - stsci.tools v 3.5 is the last version to contain it.
  warnings.warn("NMPFIT is deprecated - stsci.tools v 3.5 is the last version to contain it.")
/Users/jaguilar/Projects/miniconda3/envs/stiscoron/lib/python3.13/site-packages/stsci/tools/gfit.py:18: UserWarning: GFIT is deprecated - stsci.tools v 3.4.12 is the last version to contain it.Use astropy.modeling instead.
  warnings.warn("GFIT is deprecated - stsci.tools v 3.4.12 is the last version to contain it."


In [12]:
from astropy.io import fits
fits.info(file_managers['wav'].iloc[0]['filepath'])

Filename: /Users/jaguilar/Projects/Research/hst17092-stis_coron/coronspec_tools/example_notebooks/fringe_flat_correction/../../../data/MAST_2025-12-16T2001/HST/of0i01010/of0i01010_wav.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     224   ()      
  1  SCI           1 ImageHDU       120   (1062, 1044)   int16 (rescales to uint16)   
  2  ERR           1 ImageHDU        62   ()      
  3  DQ            1 ImageHDU        45   ()      
  4  SCI           2 ImageHDU       120   (1062, 1044)   int16 (rescales to uint16)   
  5  ERR           2 ImageHDU        62   ()      
  6  DQ            2 ImageHDU        47   ()      


In [13]:
all_files = pd.concat({k: fm[['OBSET_ID', 'filestem', 'FILENAME', 'filepath', 'TARGNAME', 'FRNGFLAT']] for k, fm in file_managers.items()}).reset_index(names=['file_type', 'file_index'])

In [14]:
sci_stem = 'of0i01030'

In [15]:
sci_file_meta = all_files.query("filestem == @sci_stem and file_type == 'raw'").squeeze()

flat_file = Path(all_files.query(f"filestem == '{sci_file_meta['FRNGFLAT'].lower()}' and file_type == 'raw'").squeeze()['filepath'])
sci_file = Path(sci_file_meta['filepath'])
wavecal_file = Path(all_files.query("filestem == @sci_stem and file_type == 'wav'").squeeze()['filepath'])

sci_file, flat_file, wavecal_file = sci_file.resolve(), flat_file.resolve(), wavecal_file.resolve()

In [17]:
str(flat_file.name).replace("raw","nsp")

'of0i01040_nsp.fits'

In [21]:
# stistools.defringe.normspflat(
#     str(flat_file),
#     str(flat_file.name.replace("raw","nsp")),
#     do_cal=True,
#     wavecal=f"{sci_file}_wav.fits"
# )


In [23]:
# file_managers['flt'].query("filestem == @filestem")['FRNGFLAT']

In [24]:
import shutil

In [25]:
local_sci_file = shutil.copy(str(sci_file), sci_file.name)
local_flat_file = shutil.copy(str(flat_file), flat_file.name)
local_wavecal_file = shutil.copy(str(wavecal_file), wavecal_file.name)

In [26]:
! rm output/*

drj_file = defringing_tools.defringe_raw2d(
    str(local_sci_file),
    str(local_flat_file),
    str(local_wavecal_file),
    output_dir='./output/'
)

rm: output/*: No such file or directory
File written:  /Users/jaguilar/Projects/Research/hst17092-stis_coron/coronspec_tools/example_notebooks/fringe_flat_correction/output/of0i01040_crj.fits
    
    *** CALSTIS-0 -- Version 3.4.2 (19-Jan-2018) ***
    Begin    17-Dec-2025 09:37:46 EST
    
    Input    of0i01030_raw.fits
    Outroot  output/of0i01030_raw.fits
    Wavecal  of0i01030_wav.fits
    
    *** CALSTIS-1 -- Version 3.4.2 (19-Jan-2018) ***
    Begin    17-Dec-2025 09:37:47 EST
    Input    of0i01030_raw.fits
    Output   output/of0i01030_blv_tmp.fits
    OBSMODE  ACCUM
    APERTURE 52X0.2F1
    OPT_ELEM G750L
    DETECTOR CCD
    
    Imset 1  Begin 09:37:47 EST
    Epcfile  of0i01bxj_epc.fits
    Warning  EPCTAB `of0i01bxj_epc.fits' not found.
    
    CCDTAB   oref$16j1600do_ccd.fits
    CCDTAB   PEDIGREE=GROUND
    CCDTAB   DESCRIP =Updated amp=D gain=4 atodgain and corresponding readnoise values---
    CCDTAB   DESCRIP =Oct. 1996 Air Calibration
    
    DQICORR  PERFORM


In [70]:
def process_row(row, **kwargs):
    filestem = row['filestem']
    sci_file = Path(row['filepath']).resolve()
    flat_file = Path(all_files.query(f"filestem == '{row['FRNGFLAT'].lower()}' and file_type == 'raw'").squeeze()['filepath']).resolve()
    wavecal_file = Path(all_files.query("filestem == @filestem and file_type == 'wav'").squeeze()['filepath']).resolve()
    local_sci_file = shutil.copy(str(sci_file), sci_file.name)
    local_flat_file = shutil.copy(str(flat_file), flat_file.name)
    local_wavecal_file = shutil.copy(str(wavecal_file), wavecal_file.name)
    drj_file = defringing_tools.defringe_raw2d(
        str(local_sci_file),
        str(local_flat_file),
        str(local_wavecal_file),
        output_dir='./output/',
        **kwargs
    )
    return drj_file

In [28]:
# ! rm output/*
# all_files.query("file_type == 'raw' and FRNGFLAT != 'N/A'").apply(process_row, axis=1)

## make 1d spectra

In [35]:
file_managers['drj'][['APERTURE', 'TARGNAME']]

,APERTURE,TARGNAME
0,52X0.2,TYC-1262-187-1
1,52X0.2F1,TYC-1262-187-1
2,52X0.2F1,TYC-1262-187-1
3,52X0.2,HD-283593
4,52X0.2F1,HD-283593
5,52X0.2F1,HD-283593
6,52X0.2,HD-114174
7,52X0.2F1,HD-114174
8,52X0.2,HD-115692
9,52X0.2F1,HD-115692


In [86]:
%matplotlib osx

In [132]:
file_managers['raw'].query("APERTURE == '52X0.2'").iloc[1]['ROOTNAME']

'of0i02010'

In [142]:
! rm output/*
! rm ./*.fits
parameters = dict(
        beg_shift=-6,
        end_shift=-6,
        shift_step=0.01,
        beg_scale=0.1,
        end_scale=2,
        scale_step=0.01,
    )
# dict(
#     beg_shift=-10,
# 	end_shift=-2,
# 	shift_step=0.05,
#     beg_scale=0.8,
# 	end_scale=1.7,
# 	scale_step=0.02,
# )
process_row(
    file_managers['raw'].query("APERTURE == '52X0.2'").iloc[2], 
    compare_spectra=True,
    **parameters
)

File written:  /Users/jaguilar/Projects/Research/hst17092-stis_coron/coronspec_tools/example_notebooks/fringe_flat_correction/output/of0i03030_crj.fits
mkfringeflat.py version 0.1
 - matching fringes in a flatfield to those in science data
 Extraction center: row 604
   Extraction size: 11.0 pixels  [Aperture: 52X0.2]
Range to be normalized: [599:610,4:1020]

Determining best shift for fringe flat

shift =     -6.000, rms =   0.4245
 
You are advised to try again after adjusting the shift range accordingly
 
 Best shift :     -6.000 pixels
 Shifted flat : output/of0i03030_nsp_sh.fits
                (Can be used as input flat for next iteration)

Determining best scaling of amplitude of fringes in flat

Fringes scaled       0.100: RMS =   0.4348
Fringes scaled       0.110: RMS =   0.4346
Fringes scaled       0.120: RMS =   0.4345
Fringes scaled       0.130: RMS =   0.4343
Fringes scaled       0.140: RMS =   0.4342
Fringes scaled       0.150: RMS =   0.4340
Fringes scaled       0.160: R

'output/of0i03010_drj.fits'

In [137]:
file_managers['raw'].query("APERTURE == '52X0.2'").iloc[2]['ROOTNAME']

'of0i03010'

In [114]:
good_params = {
   'of0i01010': dict(
        beg_shift=2.5,
        end_shift=4.9,
        shift_step=0.1,
        beg_scale=0.8,
        end_scale=1.7,
        scale_step=0.02,
    ),
    'of0i03010': dict(
        beg_shift=-9,
        end_shift=-3,
        shift_step=0.01,
        beg_scale=0.5,
        end_scale=1.7,
        scale_step=0.01,
    )
}